In [2]:
# =====================================================================
# CELL 1: SETUP, DRIVE MOUNT, AND DEPENDENCIES
# =====================================================================
from google.colab import drive
import os
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import torchvision.models as models
from collections import Counter

print("Connecting to your folder...")
drive.mount('/content/drive', force_remount=True)

# Your confirmed exact path
DATA_PATH = '/content/drive/MyDrive/Machine Learning 7 Satellite Imagery'

# Verify if GPU acceleration is active in Colab
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using computational backend device: {device}")


Connecting to your folder...
Mounted at /content/drive
Using computational backend device: cpu


In [3]:
# =====================================================================
# CELL 2: GLOBAL DATA DICTIONARY INGESTION
# =====================================================================
print("\nLoading relational mapping structures from Drive...")
with open(os.path.join(DATA_PATH, 'USGSquestions.json'), 'r') as f:
    master_questions = json.load(f)['questions']

with open(os.path.join(DATA_PATH, 'USGSanswers.json'), 'r') as f:
    master_answers = json.load(f)['answers']

with open(os.path.join(DATA_PATH, 'USGSimages.json'), 'r') as f:
    master_images = json.load(f)['images']

# Swift hash look-ups to bypass slow iterations
questions_dict = {q['id']: q for q in master_questions}
answers_dict = {a['id']: a for a in master_answers}
images_dict = {i['id']: i for i in master_images}

# Build Word & Label Vocabularies
unique_answers = sorted(list(set([a['answer'] for a in master_answers])))
answer_to_idx = {ans: i for i, ans in enumerate(unique_answers)}
idx_to_answer = {i: ans for ans, i in answer_to_idx.items()}

all_words = []
for q in master_questions:
    all_words.extend(q['question'].lower().replace('?', '').split())
word_counts = Counter(all_words)
vocab = {word: i+2 for i, (word, count) in enumerate(word_counts.items()) if count > 1}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

def tokenize_question(text, max_len=30):
    words = text.lower().replace('?', '').split()
    tokens = [vocab.get(w, 1) for w in words]
    if len(tokens) < max_len:
        tokens += [0] * (max_len - len(tokens))
    else:
        tokens = tokens[:max_len]
    return torch.tensor(tokens, dtype=torch.long)


Loading relational mapping structures from Drive...


In [4]:
# =====================================================================
# CELL 3: SATELLITE PYTORCH DATASET ENGINE
# =====================================================================
class RSVQA_USGS_Dataset(Dataset):
    def __init__(self, split_questions_file):
        with open(os.path.join(DATA_PATH, split_questions_file), 'r') as f:
            self.split_pairs = json.load(f)['questions']

        self.transform = transforms.Compose([
            transforms.Resize((112, 112)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        # Capped safely at 5,000 items for a snappy, guaranteed training run tonight
        return min(len(self.split_pairs), 5000)

    def __getitem__(self, idx):
        pair_info = self.split_pairs[idx]
        pair_id = pair_info['id']

        question_data = questions_dict[pair_id]
        answer_data = answers_dict[pair_id]
        image_data = images_dict[question_data['img_id']]

        # --- FIXED FOR RSVQA MASTER SCHEMA ---
        if 'original_name' in image_data:
            img_filename = image_data['original_name']
        elif 'filename' in image_data:
            img_filename = image_data['filename']
        else:
            img_filename = image_data.get('name', 'missing_image.tif')
        # -------------------------------------

        img_path = os.path.join(DATA_PATH, img_filename)

        if not os.path.exists(img_path):
            img_path = os.path.join(DATA_PATH, 'Images', img_filename)

        if not os.path.exists(img_path):
            image_tensor = torch.zeros((3, 112, 112)) # Safe black fallback if images are still syncing
        else:
            image = Image.open(img_path).convert('RGB')
            image_tensor = self.transform(image)

        tokenized_q = tokenize_question(question_data['question'])
        label_idx = answer_to_idx[answer_data['answer']]

        return image_tensor, tokenized_q, torch.tensor(label_idx, dtype=torch.long)

        # Fallback check just in case the images are nested inside an internal "Images" folder
        if not os.path.exists(img_path):
            img_path = os.path.join(DATA_PATH, 'Images', img_filename)

        if not os.path.exists(img_path):
            image_tensor = torch.zeros((3, 112, 112)) # Safe black image fallback if drive sync is lagging
        else:
            image = Image.open(img_path).convert('RGB')
            image_tensor = self.transform(image)

        tokenized_q = tokenize_question(question_data['question'])
        label_idx = answer_to_idx[answer_data['answer']]

        return image_tensor, tokenized_q, torch.tensor(label_idx, dtype=torch.long)

train_dataset = RSVQA_USGS_Dataset('USGS_split_train_questions.json')
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [6]:
# =====================================================================
# CELL 4: SINGLE-MODALITY TEXT-ONLY BASELINE MODEL
# =====================================================================
class TextOnlyBaselineLSTM(nn.Module):
    def __init__(self, vocab_size, num_classes):
        super(TextOnlyBaselineLSTM, self).__init__()
        # This baseline ignores images completely to fulfill the course requirements
        self.embedding = nn.Embedding(vocab_size, 128)
        self.lstm = nn.LSTM(128, 256, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, images, questions):
        # The 'images' variable is passed in by the DataLoader, but we intentionally
        # do not use it here so the model learns strictly from text frequencies.
        txt_emb = self.embedding(questions)
        _, (hn, _) = self.lstm(txt_emb)
        txt_feats = hn.squeeze(0)
        return self.classifier(txt_feats)

# Instantiate this baseline model so it's ready to train
baseline_text_model = TextOnlyBaselineLSTM(vocab_size=len(vocab), num_classes=len(unique_answers)).to(device)
baseline_criterion = nn.CrossEntropyLoss()
baseline_optimizer = torch.optim.Adam(baseline_text_model.parameters(), lr=0.001)

print("✅ Mandatory Single-Modality Text Baseline successfully compiled.")

✅ Mandatory Single-Modality Text Baseline successfully compiled.


In [7]:
# =====================================================================
# CELL 5: DEEP MULTIMODAL MODEL ARCHITECTURE
# =====================================================================
class RemoteSensingVQAEngine(nn.Module):
    def __init__(self, vocab_size, num_classes):
        super(RemoteSensingVQAEngine, self).__init__()
        # Load pre-trained ResNet18 model
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        self.img_proj = nn.Linear(512, 256)

        self.embedding = nn.Embedding(vocab_size, 128)
        self.lstm = nn.LSTM(128, 256, batch_first=True)

        self.classifier = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, images, questions):
        img_feats = self.backbone(images).squeeze(-1).squeeze(-1)
        img_emb = self.img_proj(img_feats)

        txt_emb = self.embedding(questions)
        _, (hn, _) = self.lstm(txt_emb)
        txt_feats = hn.squeeze(0)

        fused_context = torch.cat([img_emb, txt_feats], dim=1)
        return self.classifier(fused_context)

model = RemoteSensingVQAEngine(vocab_size=len(vocab), num_classes=len(unique_answers)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 173MB/s]


In [8]:
# =====================================================================
# CELL 6: BASELINE TRAINING PROCESS WITH LIVE PROGRESS PRINTING
# =====================================================================
print("\nInitiating training cycles...")
model.train()

for epoch in range(3):
    running_loss = 0.0
    batch_count = 0

    for images, questions, labels in train_loader:
        images, questions, labels = images.to(device), questions.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images, questions)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        batch_count += 1

        # --- NEW: Live update every 10 batches so you know it's alive! ---
        if batch_count % 10 == 0:
            print(f"  -> Processing Batch {batch_count}/{len(train_loader)} in Epoch {epoch+1}...")
        # -----------------------------------------------------------------

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"✅ Epoch [{epoch+1}/3] completed successfully! Cross-Entropy Loss: {epoch_loss:.4f}\n")

# Save the trained parameters for your teammates
torch.save(model.state_dict(), 'baseline_vqa_model.pth')
print("🎉 Phase 1 Complete! Engine successfully generated 'baseline_vqa_model.pth'")


Initiating training cycles...
  -> Processing Batch 10/79 in Epoch 1...
  -> Processing Batch 20/79 in Epoch 1...
  -> Processing Batch 30/79 in Epoch 1...
  -> Processing Batch 40/79 in Epoch 1...
  -> Processing Batch 50/79 in Epoch 1...
  -> Processing Batch 60/79 in Epoch 1...
  -> Processing Batch 70/79 in Epoch 1...
✅ Epoch [1/3] completed successfully! Cross-Entropy Loss: 2.6186

  -> Processing Batch 10/79 in Epoch 2...
  -> Processing Batch 20/79 in Epoch 2...
  -> Processing Batch 30/79 in Epoch 2...
  -> Processing Batch 40/79 in Epoch 2...
  -> Processing Batch 50/79 in Epoch 2...
  -> Processing Batch 60/79 in Epoch 2...
  -> Processing Batch 70/79 in Epoch 2...
✅ Epoch [2/3] completed successfully! Cross-Entropy Loss: 1.9589

  -> Processing Batch 10/79 in Epoch 3...
  -> Processing Batch 20/79 in Epoch 3...
  -> Processing Batch 30/79 in Epoch 3...
  -> Processing Batch 40/79 in Epoch 3...
  -> Processing Batch 50/79 in Epoch 3...
  -> Processing Batch 60/79 in Epoch 3.